# **Ingest drivers.Json File**
1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve nested structure)
3. Add Metadata Columns
    .Source File
    .Ingestion Time Stamp 
3. Write to bronze delta table

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
source_file_name = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

#### **Step 1 Define and enforce schema (preserve nested structure)**

In [0]:
#Define the Nested Schema 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

name_schema = StructType(fields=[
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True)
])

drivers_schema = StructType(fields=[
    StructField("driverId", StringType(), True),
    StructField("name", name_schema),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])






#### **Step-2 Read the JSon file using the dataframe reader API**

In [0]:
drivers_df = (
    spark.read
    .format("json")
#   .option("inferSchema", True)
    .schema(drivers_schema)
    .option("header", True)
    .option('mode','FAILFAST')
    .load(source_file_name)
)

#### **Step-3. Add Metadata Columns**

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
display(drivers_final_df)

####  **Step 3. Write to bronze delta table**

In [0]:
(
    drivers_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))